In [ ]:
from deepcell.applications import Mesmer
from deepcell.utils.plot_utils import create_rgb_image, make_outline_overlay
import numpy as np
import cv2
import math 
import pandas as pd
import scanpy as sc
import skimage
import imagecodecs
import tifffile as tf
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from skimage import restoration
import distinctipy
from pathlib import Path 
import pyarrow
import dask
import dask.array as da
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
from skimage.exposure import rescale_intensity
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import matplotlib.pyplot as plt
import scipy
import os
import igraph
import leidenalg

In [ ]:
os.environ.update({"DEEPCELL_ACCESS_TOKEN": ""})

In [ ]:
structure_marker_dict = {'Nuclei': ['DAPI']}
cell_type_marker_dict = {
    'Podocyte':['PODXL'],
    'Epithelial cells': ['PANCK'], 
    'Endothelial cells': ['CD31'],
    'Proximal tubule cells': ['LRP2'], 
    'B-cells': ['CD20'],
    'Macrophages': ['CD68'],
    'CD8+ T-cells': ['CD8'],
    'CD4+ T-cells': ['CD4']
    }

In [ ]:
def construct_marker_channel_dict(marker_list):
    marker_channel_dict = {}
    for marker in marker_list:
        marker_channel_dict[marker] = marker_list.index(marker)

    return marker_channel_dict

In [ ]:
all_marker_list = ['DAPI', 'PODXL', 'LRP2', 'PANCK', 'CD20', 'CD68']
marker_list = ['DAPI', 'PODXL', 'LRP2', 'PANCK', 'CD20', 'CD68']

In [ ]:
marker_channel_dict = construct_marker_channel_dict(marker_list)

In [ ]:
def normalize_img(chunk, out_range, in_range):

    normalized_img = rescale_intensity(chunk, in_range=in_range, out_range=out_range)
    return normalized_img

In [ ]:
def threshold_img(img, value, img_max):
    img_to_thr = img.copy()

    for channel in range(img_to_thr.shape[0]):
        img_channel = img_to_thr[channel, :, :]
        _, channel_mask = cv2.threshold(img_channel, 0, img_max, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        img_channel[channel_mask==0] = value
        img_to_thr[channel, :, :] = img_channel 

    return img_to_thr

In [ ]:
def extract_img_channels(img_layer, structure, img_range, cell=False):
    extracted_img_channels = []

    if(cell):
        channel_markers = cell_type_marker_dict[structure]
    else:
        channel_markers = structure_marker_dict[structure]
    for marker in channel_markers:
        img_channel_for_marker = img_layer.data[marker_channel_dict[marker], :, :]
        normalized_img_channel =normalize_img(img_channel_for_marker, img_range)
        extracted_img_channels.append(normalized_img_channel)
    
    combined_img_channels = extracted_img_channels[0]

    for channel_index in range(1, len(extracted_img_channels)):
        combined_img_channels = cv2.add(combined_img_channels, extracted_img_channels[channel_index])

    normalized_combined_img_channels = normalize_img(combined_img_channels, img_range)
    
    return normalized_combined_img_channels

In [ ]:
def format_img_for_segmentation(nuclei_channel_chunk, membrane_channel_chunk=None, nuclei=False):
    if(nuclei):
        empty_membrane_channel_chunk = da.from_array(np.zeros(nuclei_channel_chunk.shape, dtype=np.float32))
        combined_nuclei_membrane_chunk = da.stack([nuclei_channel_chunk, empty_membrane_channel_chunk], axis=-1)
    else:
        combined_nuclei_membrane_chunk = da.stack([nuclei_channel_chunk, membrane_channel_chunk], axis=-1)
    
    return combined_nuclei_membrane_chunk


In [ ]:
def cell_segmentation(combined_nuclei_membrane_chunk, segmentation_obj, nuclei=False, distance=1):

    expanded_combined_nuclei_membrane_chunk = da.expand_dims(combined_nuclei_membrane_chunk, axis=0)
    print(expanded_combined_nuclei_membrane_chunk.shape)
    if(nuclei==True):
        segmented_chunk = skimage.segmentation.expand_labels(segmentation_obj.predict(expanded_combined_nuclei_membrane_chunk, image_mpp=0.5), distance)

    else:
        segmented_chunk = segmentation_obj.predict(expanded_combined_nuclei_membrane_chunk, image_mpp=0.5)
    

    return segmented_chunk
  


In [ ]:
def merge_cell_labels(labels, tile_h, tile_w, min_contact=5):
 
    uf = nx.utils.UnionFind()
    nrows = labels.shape[0] // tile_h
    ncols = labels.shape[1] // tile_w

    for row in range(nrows):
        for col in range(ncols):
            y0, x0 = row * tile_h, col * tile_w
            tile = labels[y0:y0+tile_h, x0:x0+tile_w]

            if col < ncols - 1:
                right_tile = labels[y0:y0+tile_h, (col+1)*tile_w:(col+2)*tile_w]
                a_edge, b_edge = tile[:, -1], right_tile[:, 0]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

            if row < nrows - 1:
                bottom_tile = labels[(row+1)*tile_h:(row+2)*tile_h, x0:x0+tile_w]
                a_edge, b_edge = tile[-1, :], bottom_tile[0, :]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

    mapping, next_id = {}, 1
    for lbl in np.unique(labels):
        if lbl == 0:
            continue
        root = uf[lbl]
        if root not in mapping:
            mapping[root] = next_id
            next_id += 1
    remap = {lbl: mapping[uf[lbl]] for lbl in np.unique(labels) if lbl != 0}

    merged = labels.copy()
    mask = merged > 0
    merged[mask] = np.vectorize(remap.get)(merged[mask])

    return merged


In [ ]:
def find_cell_locations_merged(segmented_img):
    y_coords, x_coords = np.indices(segmented_img.shape)
    flat_labels = segmented_img.ravel()
    flat_x = x_coords.ravel()
    flat_y = y_coords.ravel()

    mask_nonzero = flat_labels != 0
    flat_labels_nz = flat_labels[mask_nonzero]
    flat_x_nz = flat_x[mask_nonzero]
    flat_y_nz = flat_y[mask_nonzero]


    sum_x = np.bincount(flat_labels_nz, weights=flat_x_nz)
    sum_y = np.bincount(flat_labels_nz, weights=flat_y_nz)
    counts = np.bincount(flat_labels_nz)

    labels = np.nonzero(counts)[0]
    centroids_x = sum_x[labels] / counts[labels]
    centroids_y = sum_y[labels] / counts[labels]

    cell_id_coord_dict = {lbl: (cx, cy) for lbl, cx, cy in zip(labels, centroids_x, centroids_y)}

    return cell_id_coord_dict

In [ ]:
def create_cell_marker_expression_matrix_merged(segmented_img, non_nuclei_marker_img):

    segmented_img_flattened = segmented_img.flatten()
    print('finding locations begins')
    cell_id_coord_dict = find_cell_locations_merged(segmented_img)
    
    print('locations found')
    coord_x_list = []
    coord_y_list = []
    cell_id_list = []

    for cell_id, (coord_x, coord_y) in cell_id_coord_dict.items():
        cell_id_list.append(cell_id)
        coord_x_list.append(coord_x)
        coord_y_list.append(coord_y)
    non_nuclei_marker_img_reshaped = non_nuclei_marker_img.reshape((non_nuclei_marker_img.shape[0], non_nuclei_marker_img.shape[1]*non_nuclei_marker_img.shape[2]))

    cell_type_markers = marker_list[1:] #[marker for marker_list in list(cell_type_marker_dict.values()) for marker in marker_list]
    key_marker_list = []

    cell_marker_expression_dict = {'Cell ID': segmented_img_flattened}
    for marker, channel_index in marker_channel_dict.items():
        if(marker in cell_type_markers):
            cell_marker_expression_dict[marker] = non_nuclei_marker_img_reshaped[channel_index-1]
            key_marker_list.append(marker)
    cell_marker_expression_df = pd.DataFrame(cell_marker_expression_dict) 
    agg_function_dict = {marker:'mean' for marker in key_marker_list}

    cell_marker_expression_df_grouped_by_cell = pd.DataFrame(cell_marker_expression_df.groupby('Cell ID').agg(agg_function_dict)) 
    
    cell_location_df = pd.DataFrame({'Cell ID': cell_id_list, 'X':coord_x_list, 'Y': coord_y_list}) 
                            
    cell_expression_location_df = pd.merge(cell_marker_expression_df_grouped_by_cell, cell_location_df, on='Cell ID')
    
    cell_expression_location_df = cell_expression_location_df[cell_expression_location_df['Cell ID'] != 0]

    return cell_expression_location_df, cell_location_df, cell_marker_expression_df, cell_marker_expression_df_grouped_by_cell


In [ ]:
def cluster_cells(cell_expression_location_df, cluster_alg, max_val, num_neighbors, res): 
    
    cell_expression_location_lower_quantile = cell_expression_location_df.quantile(q=0.02,axis=0)
    
    for col in list(cell_expression_location_df.columns.values):
        if col not in ['Cell ID','X','Y']: 
            cell_expression_location_df.loc[cell_expression_location_df[col]<cell_expression_location_lower_quantile[col], col] = 0

    cell_marker_expression_df = cell_expression_location_df.copy(deep=True).drop(columns=['X', 'Y', 'Cell ID']) 

    cell_marker_expression_obj = sc.AnnData(cell_marker_expression_df)
    cell_marker_expression_obj.var_names_make_unique()
    sc.pp.scale(cell_marker_expression_obj, max_value=max_val, zero_center=False)

    sc.pp.neighbors(cell_marker_expression_obj, n_neighbors=num_neighbors)

    print('preparing for clustering')

    if(cluster_alg=='louvain'):
        sc.tl.louvain(cell_marker_expression_obj, resolution=res)
        cell_location_expression_cluster_df = pd.concat([cell_expression_location_df.reset_index(), cell_marker_expression_obj.obs.reset_index()], axis=1)
        cell_location_expression_cluster_df = cell_location_expression_cluster_df.rename(columns={'louvain':'Cluster'})
    else:
        sc.tl.leiden(cell_marker_expression_obj, resolution=res)
        cell_location_expression_cluster_df = pd.concat([cell_expression_location_df.reset_index(), cell_marker_expression_obj.obs.reset_index()], axis=1)
        cell_location_expression_cluster_df = cell_location_expression_cluster_df.rename(columns={'leiden':'Cluster'})
    
    print('clustering')

    cluster_cell_type_dict = {}
    cell_type_dict = {}
    for cluster in list(cell_location_expression_cluster_df['Cluster'].unique()):
        mean_expression_cluster_df = cell_location_expression_cluster_df[cell_location_expression_cluster_df['Cluster'] == cluster].describe()
        cell_type_mean_expression_dict = {}
        for cell_type, marker_list in cell_type_marker_dict.items():
            marker_list_mean_expression =  np.array([mean_expression_cluster_df.loc['mean', marker] for marker in marker_list]).flatten()
            cell_type_mean_expression_dict[cell_type] = np.mean(marker_list_mean_expression)
        cluster_cell_type_dict[cluster] = list(cell_type_mean_expression_dict.keys())[list(cell_type_mean_expression_dict.values()).index(max(list(cell_type_mean_expression_dict.values())))]
        if(cell_type_dict.get(cluster_cell_type_dict[cluster])!=None):
            cell_type_dict[cluster_cell_type_dict[cluster]].append(cluster)
        else:
            cell_type_dict[cluster_cell_type_dict[cluster]] = [cluster]
    
    print('assigning cluster')

    cluster_cell_type_dict_for_df = {'Cluster':[], 'Cell Type': []}
    for cluster, cell in cluster_cell_type_dict.items():
        cluster_cell_type_dict_for_df['Cluster'].append(cluster)
        cluster_cell_type_dict_for_df['Cell Type'].append(cell)
    
    cell_location_expression_cluster_type_df = pd.merge(cell_location_expression_cluster_df, pd.DataFrame(cluster_cell_type_dict_for_df), on='Cluster', how='left')


    return cell_location_expression_cluster_type_df, cluster_cell_type_dict

In [ ]:
def create_color_dict(cell_type_list):
    color_arr = distinctipy.get_colors(len(cell_type_list))
    cell_type_color_dict = {}
    color_index = 0
    for i in range(len(cell_type_list)):
        cell_type_color_dict[cell_type_list[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
def view_clustered_cells_merged(cell_location_expression_cluster_type_df, cell_type_color_dict, merged_segmentation_mask_matrix, img_h, img_w, circle=True, background=None):

    if(background == None):
        img = np.full((img_h, img_w, 3), (0, 0, 0))

    if(circle == True):
        for cell_type, group in cell_location_expression_cluster_type_df.groupby('Cell Type'):
            r, g, b = cell_type_color_dict[cell_type]
            color = (int(r * 255), int(g * 255), int(b * 255))
            pts = group[['X', 'Y']].to_numpy(dtype=np.int32)
            for x, y in pts:
                cv2.circle(img, (x, y), 10, color, -1, lineType=cv2.LINE_AA)
    else:
        labels = merged_segmentation_mask_matrix

        for cell_type, (r, g, b) in cell_type_color_dict.items():
            color = np.array(
                [int(r * 255), int(g * 255), int(b * 255)],
                dtype=np.uint8
            )

            cell_idx = (
                cell_location_expression_cluster_type_df
                .loc[cell_location_expression_cluster_type_df['Cell Type'] == cell_type, 'index']
                .astype(int)
                .values
            )

            lookup = np.zeros(labels.max() + 1, dtype=bool)
            lookup[cell_idx] = True

            mask = lookup[labels]
            img[mask] = color


    return img
 

## Run Cell Segmentation and Clustering

In [ ]:
thr_dict_exists = False 
thr_dict_suffix = ''
cell_clustering_repeat = False 

In [ ]:
base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering/Multiplex 2/ANCA 9/1x'

In [ ]:
Path.mkdir(Path('/'.join(base_path.split('/')[:-2])), exist_ok=True)

In [ ]:
Path.mkdir(Path('/'.join(base_path.split('/')[:-1])), exist_ok=True)

In [ ]:
Path.mkdir(Path(base_path), exist_ok=True)

In [ ]:
IF_channels_path = Path('Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 2/ANCA 9')

In [ ]:
CH1_found = False
channel_paths = [f for f in IF_channels_path.rglob('*.tif') if 'registered' in str(f) and 'Cycle' in str(f)]

channels = {}
i = 0
j = 0
for channel in channel_paths:
    print(channel)
    if('CH1' in str(channel)):
        if(CH1_found == True):
            continue 
        else:
            CH1_found = True
    if(all_marker_list[i] in marker_list):
        channels[marker_list[j]] = da.from_array(tf.imread(channel), chunks=(512, 512))
        j+=1 
    i+=1 
    

In [ ]:
with open('/'.join(str(IF_channels_path).split('\\')) + '/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
def resize_chunk(chunk, factor):
    return cv2.resize(chunk, None, fx=1/factor, fy=1/factor)

In [ ]:
channel_size = []
for channel_name, channel in channels.items(): 
    channel_size.append(channel.shape)

In [ ]:
assert thr_dict_exists, "create threshold dictionary"


if(cell_clustering_repeat):
    thr_dict_path = '/'.join(base_path.split('/')[:-1])
    with open(thr_dict_path +  '/thr_dict' + thr_dict_suffix + '.pickle', 'rb') as f: 
        thr_dict = pickle.load(f)
else:
    with open(base_path +  '/thr_dict' + thr_dict_suffix + '.pickle', 'rb') as f: 
        thr_dict = pickle.load(f)


In [ ]:
channels_normalized = {}
for channel_name, channel in channels.items():
    channels_normalized[channel_name] = channel.map_blocks(normalize_img, (0, 255), (da.min(channel).compute(), da.max(channel).compute()))

In [ ]:
def threshold_image(chunk, channel):
   
    if(channel == 'DAPI'):
        return chunk
    else:
        chunk_mask = chunk > thr_dict[channel]
    chunk_thr = chunk.copy()
    chunk_thr[chunk_mask==0] = 0

    return chunk_thr 

In [ ]:
channels_thresholded = {}
for channel_name, channel in channels_normalized.items():
    print(channel_name)
    channels_thresholded[channel_name] = channel.map_blocks(threshold_image, channel_name) 

In [ ]:
non_nuclei_marker_stack = da.stack([channels_thresholded[t] for t in list(channels_thresholded.keys()) if t!='DAPI'], axis=0) 
non_nuclei_marker_stack = non_nuclei_marker_stack.rechunk(chunks=(len(marker_channel_dict.keys())-1, 512, 512), balance=True)

In [ ]:
nuclei_img =  channels_normalized['DAPI'] 

In [ ]:
tile_combined_location_img_list = []
for i in range(0, nuclei_img.shape[0], 512):
    for j in range(0, nuclei_img.shape[1], 512):
        tile_combined_location_img_list.append((i, j))


In [ ]:
nuclei_img_for_segmentation = nuclei_img.map_blocks(format_img_for_segmentation, nuclei=True, chunks=(512, 512, 2), new_axis=[2])

In [ ]:
segmentation_obj = Mesmer()

In [ ]:
segmented_img = nuclei_img_for_segmentation.map_blocks(cell_segmentation, segmentation_obj, True, 1, dtype=np.int64, chunks=(1, 512, 512, 1), new_axis=[0])

In [ ]:
segmented_img_chunks = segmented_img.blocks.ravel()

In [ ]:
def compute_overlay_chunk(chunk_idx, chunk_list):
    return np.squeeze(chunk_list[chunk_idx].compute())

In [ ]:
segmented_img_chunks_computed = [compute_overlay_chunk(block_idx, segmented_img_chunks) for block_idx in range(len(segmented_img_chunks))]

In [ ]:
segmentation_mask = []
img_h = nuclei_img_for_segmentation.shape[0]
img_w = nuclei_img_for_segmentation.shape[1]
tile_h = 512
tile_w = 512
j = 0
for i in range(int(img_h/tile_h)):
    segmentation_mask.append(np.hstack(segmented_img_chunks_computed[j:j+int(img_w/tile_w)]))
    j+= int(img_w/tile_w)
        
segmentation_mask_matrix = np.vstack(segmentation_mask)

In [ ]:
non_nuclei_marker_chunks = non_nuclei_marker_stack.blocks.ravel()
non_nuclei_marker_chunks_computed = [compute_overlay_chunk(block_idx, non_nuclei_marker_chunks) for block_idx in tqdm(range(len(non_nuclei_marker_chunks)))]

In [ ]:

non_nuclei_marker_matrix = np.zeros(non_nuclei_marker_stack.shape)
img_h = non_nuclei_marker_stack.shape[1]
img_w = non_nuclei_marker_stack.shape[2]
tile_h = 512
tile_w = 512
k = 0
for i in range(0, img_h, tile_h):
    for j in range(0, img_w, tile_w):
        non_nuclei_marker_matrix[:, i:i+512, j:j+512] = non_nuclei_marker_chunks_computed[k]
        k+=1 
    

In [ ]:
segmentation_mask_offset = np.zeros(segmentation_mask_matrix.shape)

offset = 0
for l in range(len(tile_combined_location_img_list)):
    i, j = tile_combined_location_img_list[l]
    segmented_tile = segmentation_mask_matrix[i:i+512, j:j+512] 
    segmented_tile_non_background_mask = segmented_tile!=0 
    segmented_tile_offset = segmented_tile.copy()
    segmented_tile_offset[segmented_tile_non_background_mask] = segmented_tile[segmented_tile_non_background_mask] + offset
    segmentation_mask_offset[i:i+512, j:j+512] = segmented_tile_offset
    offset+=np.max(segmented_tile)


In [ ]:
import networkx as nx

def merge_cell_labels(labels, tile_h, tile_w, min_contact=5):
 
    uf = nx.utils.UnionFind()
    nrows = labels.shape[0] // tile_h
    ncols = labels.shape[1] // tile_w

    for row in range(nrows):
        for col in range(ncols):
            y0, x0 = row * tile_h, col * tile_w
            tile = labels[y0:y0+tile_h, x0:x0+tile_w]

            if col < ncols - 1:
                right_tile = labels[y0:y0+tile_h, (col+1)*tile_w:(col+2)*tile_w]
                a_edge, b_edge = tile[:, -1], right_tile[:, 0]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

            if row < nrows - 1:
                bottom_tile = labels[(row+1)*tile_h:(row+2)*tile_h, x0:x0+tile_w]
                a_edge, b_edge = tile[-1, :], bottom_tile[0, :]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

    mapping, next_id = {}, 1
    for lbl in np.unique(labels):
        if lbl == 0:
            continue
        root = uf[lbl]
        if root not in mapping:
            mapping[root] = next_id
            next_id += 1
    remap = {lbl: mapping[uf[lbl]] for lbl in np.unique(labels) if lbl != 0}

    merged = labels.copy()
    mask = merged > 0
    merged[mask] = np.vectorize(remap.get)(merged[mask])

    return merged

In [ ]:
def estimate_min_contact(labels, tile_h, tile_w, percentile=10):
    contact_counts = []

    nrows = labels.shape[0] // tile_h
    ncols = labels.shape[1] // tile_w

    for row in range(nrows):
        for col in range(ncols):
            y0, x0 = row*tile_h, col*tile_w
            tile = labels[y0:y0+tile_h, x0:x0+tile_w]

            if col < ncols - 1:
                right_tile = labels[y0:y0+tile_h, (col+1)*tile_w:(col+2)*tile_w]
                left_edge, right_edge = tile[:, -1], right_tile[:, 0]
                touching = (left_edge>0) & (right_edge>0)
                a_vals, b_vals = left_edge[touching], right_edge[touching]
                for a in np.unique(a_vals):
                    for b in np.unique(b_vals):
                        n_touch = np.sum((left_edge==a) & (right_edge==b))
                        contact_counts.append(n_touch)

            if row < nrows - 1:
                bottom_tile = labels[(row+1)*tile_h:(row+2)*tile_h, x0:x0+tile_w]
                top_edge, bottom_edge = tile[-1, :], bottom_tile[0, :]
                touching = (top_edge>0) & (bottom_edge>0)
                a_vals, b_vals = top_edge[touching], bottom_edge[touching]
                for a in np.unique(a_vals):
                    for b in np.unique(b_vals):
                        n_touch = np.sum((top_edge==a) & (bottom_edge==b))
                        contact_counts.append(n_touch)

    contact_counts = np.array(contact_counts)

    if len(contact_counts) == 0:
        return 1 

    min_contact = max(1, int(np.percentile(contact_counts, percentile)))

    return min_contact

In [ ]:
merged_segmentation_mask_matrix = merge_cell_labels(segmentation_mask_offset, 512, 512, estimate_min_contact(segmentation_mask_offset, 512, 512))

In [ ]:
merged_segmentation_mask_matrix = merged_segmentation_mask_matrix.astype(int)

In [ ]:
with open(base_path + '/merged_segmentation_mask_matrix_for_pixel_clustering.pickle', 'wb') as f:
    pickle.dump(merged_segmentation_mask_matrix, f)

In [ ]:
cell_marker_matrix_merged, cell_location_df, cell_marker_expression_df, cell_marker_expression_df_grouped_by_cell = create_cell_marker_expression_matrix_merged(merged_segmentation_mask_matrix, non_nuclei_marker_matrix)

In [ ]:
cell_marker_matrix_merged[['X', 'Y']] = cell_marker_matrix_merged[['X', 'Y']].astype(int)

In [ ]:
x = cell_marker_matrix_merged['X']
y = cell_marker_matrix_merged['Y']
cell_id = cell_marker_matrix_merged['Cell ID']

cell_marker_exp_matrix_no_meta = cell_marker_matrix_merged.drop(columns=['X', 'Y', 'Cell ID'])

In [ ]:
cell_marker_matrix_merged.to_csv(base_path + '/cell_expression_location_merged.csv')

In [ ]:
cell_cluster_loc_exp_df, cluster_cell_type_dict = cluster_cells(cell_marker_matrix_merged, 'leiden', None, 15, 0.5)

In [ ]:
cell_type_list = list(cell_cluster_loc_exp_df['Cell Type'].unique())
cell_type_color_dict = create_color_dict(cell_type_list)
cell_type_color_dict['Unclassified'] = (0.5, 0.5, 0.5)

In [ ]:
with open(base_path + '/cell_type_color_dict.pickle', 'wb') as f:
    pickle.dump(cell_type_color_dict, f)

In [ ]:
with open(base_path + '/cell_type_color_dict.pickle', 'rb') as f:
    cell_type_color_dict = pickle.load(f)

In [ ]:
cell_cluster_loc_exp_df.to_csv(base_path + '/cell_cluster_location_merged.csv')

In [ ]:
cell_cluster_loc_exp_df_unclassified = cell_cluster_loc_exp_df.copy()

In [ ]:
cell_cluster_loc_exp_df_unclassified.loc[(cell_cluster_loc_exp_df_unclassified[marker_list[1:]].le(0.025*255)).all(axis=1), 'Cell Type'] = 'Unclassified'


In [ ]:
x_coords = cell_cluster_loc_exp_df_unclassified['X'].to_numpy().astype(int)
y_coords = cell_cluster_loc_exp_df_unclassified['Y'].to_numpy().astype(int)

cell_cluster_loc_exp_df_unclassified['Cell ID'] = merged_segmentation_mask_matrix[y_coords, x_coords]
cell_cluster_loc_exp_df_unclassified = cell_cluster_loc_exp_df_unclassified[cell_cluster_loc_exp_df_unclassified['Cell ID'].duplicated(keep=False) == False]

In [ ]:
cell_segmentation_mask_binary = np.zeros(merged_segmentation_mask_matrix.shape)
cell_segmentation_mask_binary[merged_segmentation_mask_matrix!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((7, 7), np.uint8), iterations=1), np.ones((41, 41), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)


In [ ]:
merged_segmentation_mask_matrix_within_contour = merged_segmentation_mask_matrix.copy()
merged_segmentation_mask_matrix_within_contour[contour_drawn==0] = 0

In [ ]:
with open(base_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'wb') as f:
    pickle.dump(merged_segmentation_mask_matrix_within_contour, f)

In [ ]:
mask_cell_ids = np.unique(merged_segmentation_mask_matrix_within_contour)

In [ ]:
mask_cell_ids = mask_cell_ids[1:]

In [ ]:
ind_to_drop = []
for ind in cell_cluster_loc_exp_df_unclassified.index:
    cell_id = int(cell_cluster_loc_exp_df_unclassified['Cell ID'][ind])
    if(cell_id not in mask_cell_ids): 
        ind_to_drop.append(ind)

In [ ]:
cell_cluster_loc_exp_within_contour_df = cell_cluster_loc_exp_df_unclassified.drop(index=ind_to_drop)

In [ ]:
cell_cluster_loc_exp_within_contour_df.shape

In [ ]:
cell_cluster_loc_exp_within_contour_df.to_csv(base_path + '/cell_cluster_location_merged_unclassified_within_contour.csv')

In [ ]:
cell_type_overlay_matrix = view_clustered_cells_merged(cell_cluster_loc_exp_within_contour_df, cell_type_color_dict, merged_segmentation_mask_matrix_within_contour, merged_segmentation_mask_matrix_within_contour.shape[0], merged_segmentation_mask_matrix_within_contour.shape[1], True)

In [ ]:
fig, ax = plt.subplots(1, len(roi))
fig.set_figheight(30)
fig.set_figwidth(50)
i = 0
for name, coord in roi.items():
    print(name)
    img_labeled = cell_type_overlay_matrix[coord[0]:coord[1], coord[2]:coord[3]]
    ax[i].imshow(img_labeled.astype(np.uint8))
    ax[i].axis('off')
    i+=1 

plt.subplots_adjust(wspace=0.025, hspace=0.025)


In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(cell_type_color_dict.keys()), list(cell_type_color_dict.values()))]
ax.legend(patches, list(cell_type_color_dict.keys()), loc='center', frameon=False, ncol=1) #int(len(list(cell_type_color_dict.keys()))))
ax.axis('off')
i+=1
plt.show()

In [ ]:
fig, ax = plt.subplots(1)
fig.set_figheight(100)
fig.set_figwidth(25)
ax.imshow(cell_type_overlay_matrix)